# [모델] LightGBM + CatBoost 앙상블 — 학습

KBO 투구 하나가 **제구 성공** 투구일 확률을 예측합니다.

- **입력**: `test.csv` 의 47개 컬럼 + 같은 행 안의 값 또는 정적 lookup 테이블로 만든 파생 피처
- **출력**: 제구 성공 확률 (0 이상 1 이하의 실수)
- **평가지표**: Brier Skill Score

베이스라인(RandomForest)을 LightGBM 단일 모델(피처 엔지니어링 + isotonic 보정)로 교체한 뒤,
CatBoost를 추가해 **LightGBM/CatBoost 앙상블**로 다시 개선했습니다. 2024 시즌을
검증으로 떼어 둔 홀드아웃 실험에서 Validation Score 추이:

RandomForest 454 → LightGBM+피처 693 → LightGBM+isotonic 723 →
LightGBM+CatBoost 블렌드+isotonic 748 → +trackman_history 카운트 lookup 피처 766.04 →
+expanding-window 보정 재검증 748.08 → +블렌드 가중치 튜닝 750.75 →
+isotonic 보정 OOF를 KFold(shuffle)로 되돌림 766.04 → +li 파생 피처 774.07 →
**+로지스틱 회귀 메타러너 스태킹(v10, 격자탐색 블렌드+isotonic 대체) 782.47**

`trackman_history.csv`의 `pitcher_trackman_id`/`batter_trackman_id`는 `train.csv`/
`test.csv`의 `pitcher_id`/`batter_id`와 값 범위가 완전히 다르고 교집합이 없어(직접 확인:
겹치는 ID 0건), 팀 코드(`pitcher_team`, 26종)도 `pitcher_team_id`(13종)와 신뢰성 있게
대응시킬 수 없어 **선수/팀 단위 결합은 불가능**합니다. 대신 두 데이터셋에 공통으로 존재하는
카운트 상황 키(`balls_before`, `strikes_before`, `outs_before`)로 리그 lookup 테이블을
만들어 병합했습니다 (섹션 2.5, 3 참고) — 개별 선수 정보가 아니라 "이 카운트에서 리그가
보통 어떻게 던지는가" 라는 상황 정보이므로 안티리키지 규칙에 저촉되지 않습니다.

**lookup은 시즌별로 두 개를 따로 만듭니다** — `TK_LOOKUP_VAL`(시즌 ≤ 2023, 2024 홀드아웃
검증 전용)과 `TK_LOOKUP_FULL`(시즌 ≤ 2024, 최종 재학습·제출 아티팩트 전용). 처음에는
시즌 구분 없이 lookup 하나만 만들어 검증에도 그대로 썼는데, 그러면 검증 대상인 2024 시즌
자체의 Trackman 분포가 이미 lookup에 녹아 있어 "미래 예측"을 검증하는 게 아니게 된다 —
PR #3 코드 리뷰(GPT Pro)에서 지적받고 분리했다(748.36 → 770.19로 부풀어 있던 것을
765.86으로 재검증). 3차 리뷰에서 v2부터 있던 별개의 버그(`CatBoostClassifier.
get_best_iteration()`의 0-based 인덱스를 `iterations=`에 그대로 넘기던 off-by-one)도
발견돼 `+ 1` 보정 후 766.04가 됐다.

4차 리뷰에서는 isotonic 보정용 OOF가 `KFold(shuffle=True)` 무작위 분할이라 `asof_*`
(시간 누적 성적) 피처 특성상 완전한 시간순은 아니라는 지적을 받았다(issue #4). v5에서
시즌 단위 expanding-window OOF로 재설계했고, 정직하게 재검증한 점수는 766.04 → 748.08
로 낮아졌다가, v7에서 KFold(shuffle) 방식으로 되돌리면서 766.04를 회복했다(자세한 내용은
"3.5" 절, README.md의 "v7" 절 참고).

**v10 — 로지스틱 회귀 메타러너 스태킹**: v5~v9까지 써 온 "블렌드 가중치 격자탐색(W_GRID)
+ 보정 순서 격자탐색 + isotonic 보정"을 LightGBM/CatBoost의 KFold OOF 예측 2개를
피처로 하는 로지스틱 회귀 메타러너로 교체했다(섹션 6/7 참고). 메타러너의 raw
`predict_proba` 출력이 그 위에 isotonic 보정을 한 번 더 씌운 경우보다 2024 홀드아웃에서
더 좋게 나와(782.47 vs 770.21) 추가 보정 없이 그대로 쓴다. 자세한 내용은 README.md의
"v10" 절과 `docs/logistic_stack_diagnostic.py` 참고. **이 변경도 로컬 홀드아웃 점수만으로
최종 채택하지 않는다** — Dacon 재제출 실측 점수 확인 전까지는 머지 보류.

이 노트북은 저장소 루트의 `data/` 를 읽어(`DATA_DIR = "../data"`, 이 노트북이
`notebooks/` 아래에 있기 때문) 두 모델과 로지스틱 회귀 메타러너, trackman lookup
테이블을 하나의 딕셔너리로 묶어 `../model/ensemble.pkl` 로 저장합니다. 저장한 모델은
`src/script.py` 와 함께 제출용 zip으로 묶어 제출합니다 (zip 안에서는
`model/ensemble.pkl` 처럼 상대경로가 한 단계 짧아짐). `trackman_history.csv`는 학습
시점에만 lookup 테이블을 만드는 데 쓰이고, 그 결과(`tk_lookup` = `TK_LOOKUP_FULL`)가
아티팩트에 저장되므로 추론(`script.py`) 시점에는 `trackman_history.csv` 파일 자체가
필요 없습니다.

## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 모델 학습(`lightgbm`, `catboost`), 확률 보정(scikit-learn의
`IsotonicRegression`, `KFold`)에 필요한 라이브러리를 불러옵니다. `joblib` 은 학습한
모델을 파일로 저장할 때 사용합니다.

In [1]:
import os
import time

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

# 이 노트북은 notebooks/ 아래에 있으므로 저장소 루트 기준 경로는 한 단계 위(../)이다.
# (제출용 script.py는 zip 안에서 model/, script.py가 같은 위치에 있으므로 "./model" 을 쓴다 —
#  이 노트북과는 경로 기준이 다르니 script.py로 코드를 옮길 때 접두사를 다시 확인할 것.)
DATA_DIR = "../data"

ID_COL = "row_id"
TARGET = "control_success"
CAT_COLS = ["top_bottom", "game_type", "base_state"]
# v10부터는 블렌드 가중치 격자탐색(W_GRID, v5~v9) + isotonic 보정을 로지스틱 회귀
# 메타러너(스태킹)로 대체한다 — 섹션 6/7 참고. LightGBM/CatBoost의 KFold OOF 예측
# 2개를 입력으로 하는 선형 메타러너를 학습해 결합 방식 자체를 데이터로 고른다.
# (isotonic은 더 이상 쓰지 않는다 — 메타러너 raw 출력이 그 위에 isotonic을 한 번
# 더 씌운 경우보다 2024 홀드아웃에서 더 좋게 나왔다: 782.47 vs 770.21, 섹션 6 참고.)

## 2. 데이터 불러오기

`train.csv` 는 2019~2024 시즌이고 평가 데이터는 2025 시즌입니다.

사용할 피처 목록은 `test.csv` 가 정합니다. `train.csv` 에만 있는 컬럼을 학습에 넣으면
평가 시점에 그 컬럼이 없어 추론이 실패하기 때문입니다.

In [2]:
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID_COL]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])

print("train:", train.shape, "| 원본 피처:", len(FEATURES))
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")

# 검증 단계(2019~2023 학습 -> 2024 검증)의 cold-start 스무딩에 쓸 전역 평균.
# 검증 대상인 2024 시즌은 제외하고 계산해 검증 점수가 낙관적으로 부풀지 않게 한다.
GLOBAL_MEAN_VAL = train.loc[train["season"] != 2024, TARGET].mean()
print("전역 평균(2024 제외, 검증용):", GLOBAL_MEAN_VAL)

train: (1475092, 48) | 원본 피처: 47
시즌: 2019 ~ 2024
제구 성공률: 0.5238
전역 평균(2024 제외, 검증용): 0.5315815109059132


## 2.5. `trackman_history.csv` 카운트 상황별 리그 lookup 테이블

`trackman_history.csv`의 `pitcher_trackman_id`/`batter_trackman_id`는 `train.csv`/
`test.csv`의 `pitcher_id`/`batter_id`와 값 범위가 겹치지 않고(직접 확인: 교집합 0건),
`pitcher_team`(문자열 팀코드, 26종 — 올스타/군보류 팀 등 포함)도 `pitcher_team_id`
(정수 13종)와 신뢰성 있게 대응시킬 수 없어 **선수/팀 단위로는 결합할 수 없다.**

대신 두 데이터셋에 공통으로 존재하는, 투구 직전에 이미 알 수 있는
**카운트 상황 키**(`balls_before`, `strikes_before`, `outs_before`)로 2019~2024년
`trackman_history.csv` 전체를 집계해 "이 카운트에서 리그 전체가 보통 어떤 구종을
어떤 구위로 던지는가"를 나타내는 **정적 lookup 테이블**을 만든다. `train.csv`/
`test.csv`의 어느 행도 이 집계에 들어가지 않고 2019~2024 `trackman_history`만
사용하므로 안티리키지 규칙을 지킨다.

**v5에서 `same_hand`(투수·타자 좌우 매치업)를 lookup 키에 추가해봤지만 기각했다.**
같은 블렌드 가중치/보정 순서로 고정하고 ablation한 결과, 카운트 키만 쓴 경우
750.75, `same_hand`를 추가한 경우 749.34로 오히려 **-1.41 악화**됐다 — 매치업
정보는 이미 `same_hand` 단일 피처로 모델에 들어가 있어서, lookup 키에까지
추가하면 새 신호보다는 셀당 표본 수 감소(72개 조합으로 세분화되며 셀당 평균
25,000행)로 인한 노이즈가 더 크게 작용한 것으로 보인다. 그래서 카운트 상태
3개 컬럼만 키로 쓰는 v3/v4 방식을 유지한다.

**검증용과 최종 제출용 lookup을 분리한다.** 2024 시즌을 홀드아웃으로 검증할 때
lookup 테이블 자체가 2019~2024 전체(2024 포함)로 만들어져 있으면, 검증 대상인
2024 시즌의 실제 Trackman 분포가 이미 lookup에 녹아 있는 셈이라 미래 정보를
미리 아는 것과 같아진다 — 이러면 검증 점수가 낙관적으로 부풀 수 있다. 그래서
- `TK_LOOKUP_VAL` (시즌 ≤ 2023): 섹션 4~6의 2024 홀드아웃 검증에만 사용
- `TK_LOOKUP_FULL` (시즌 ≤ 2024): 섹션 7의 전체 데이터 재학습·최종 아티팩트 저장에만 사용

로 나눠서, 검증 실험은 실제 제출 시나리오(학습에 없는 미래 시즌을 예측)를 정확히
모사하게 한다.

이 투수의 평소 구종 비율(`asof_pitcher_fastball_rate` 등, 이미 제공되는 피처)과
이 lookup 값의 차이를 추가로 만들면 "이 카운트에서 리그가 보통 어떻게 던지는가
대비 이 투수가 얼마나 벗어나는가"라는 신호가 된다.

In [3]:
TK_KEYS = ["balls_before", "strikes_before", "outs_before"]

tk_raw = pd.read_csv(
    os.path.join(DATA_DIR, "trackman_history.csv"), encoding="utf-8-sig",
    usecols=["season"] + TK_KEYS + ["pitch_type_group", "rel_speed", "spin_rate",
             "induced_vert_break", "horz_break", "extension", "zone_speed"],
)
assert tk_raw["season"].between(2019, 2024).all(), "trackman_history.csv에 2019~2024 범위 밖 시즌이 있음"


def make_tk_lookup(tk_df):
    lookup = tk_df.groupby(TK_KEYS).agg(
        tk_fastball_rate=("pitch_type_group", lambda s: (s == "fastball").mean()),
        tk_breaking_rate=("pitch_type_group", lambda s: (s == "breaking").mean()),
        tk_offspeed_rate=("pitch_type_group", lambda s: (s == "offspeed").mean()),
        tk_zone_speed_mean=("zone_speed", "mean"),
        tk_rel_speed_mean=("rel_speed", "mean"),
        tk_spin_rate_mean=("spin_rate", "mean"),
        tk_horz_break_std=("horz_break", "std"),
        tk_vert_break_std=("induced_vert_break", "std"),
        tk_extension_mean=("extension", "mean"),
    ).reset_index()
    assert not lookup.duplicated(TK_KEYS).any(), "tk_lookup에 중복 카운트 키가 있음"
    return lookup


# 검증(2024 홀드아웃)에는 2024를 뺀 lookup을, 최종 제출 모델에는 전체 시즌 lookup을 쓴다
# (섹션 2.5 설명 참고 — 안 그러면 검증 대상 시즌의 정보가 lookup에 새어 들어간다).
TK_LOOKUP_VAL = make_tk_lookup(tk_raw[tk_raw["season"] <= 2023])
TK_LOOKUP_FULL = make_tk_lookup(tk_raw)
# tk_raw는 여기서 지우지 않는다 — 섹션 6/7의 isotonic 보정용 expanding-window OOF가
# fold별로 더 이른 cutoff의 lookup(예: season < 2021)을 다시 만들어야 하기 때문.

covered = train.merge(
    TK_LOOKUP_FULL[TK_KEYS].assign(_tk=1), on=TK_KEYS, how="left",
)["_tk"].notna().mean()
print("trackman count-state lookup (val, ≤2023):", TK_LOOKUP_VAL.shape,
      "| (full, ≤2024):", TK_LOOKUP_FULL.shape)
print(f"train.csv 행 중 lookup 키로 매칭되는 비율: {covered:.4%} (매칭 안 되는 행은 NaN -> 모델이 결측으로 자연 처리)")

trackman count-state lookup (val, ≤2023): (55, 12) | (full, ≤2024): (55, 12)
train.csv 행 중 lookup 키로 매칭되는 비율: 100.0000% (매칭 안 되는 행은 NaN -> 모델이 결측으로 자연 처리)


In [4]:
# =======================
# 3. 피처 엔지니어링
# =======================
# 아래 파생 피처는 모두 "같은 행 안의 값" 또는 "행의 키로 병합하는 정적 lookup
# 테이블"만 사용한다 (test.csv 내부 다른 행에 대한 집계·빈도·순서 기반 피처는 전혀
# 만들지 않는다 — 대회 규칙상 금지).
# - 카운트/레버리지: 볼카운트 조합, 승리기대값 차이, 후반 접전 여부, li(leverage index)
#   고레버리지 플래그 및 count_diff/late_and_close와의 상호작용(v8 — li는 원본
#   컬럼으로만 존재하고 파생 피처가 없었다. train.csv 분포를 직접 확인: 평균 0.98,
#   중앙값 0.80, li>=1.5는 상위 19.7%(사바메트릭스 관행 임계치 1.5와 실제 분포의
#   80~90분위 사이가 맞아떨어져 임계치로 채택), 결측 0건)
# - 매치업: 투수·타자 좌우 일치 여부, asof 성공률 차이(표본이 작으면 global_mean으로 수축)
# - cold-start 플래그: asof_*_n == 0 인 행을 모델이 별도로 구분할 수 있게 표시
# - 구종 성향: fastball/breaking/offspeed 중 최대 비중, 최근 경기 성공률 추세
# - 주기성: game_month/game_dayofweek 의 sin/cos 인코딩
# - trackman: (balls_before, strikes_before, outs_before) 카운트 상황별 리그 전체
#   구종/구속/무브먼트 lookup(tk_lookup — 호출부가 검증용/최종용을 구분해서 넘김,
#   섹션 2.5 참고)을 키로 병합하고, 이 투수의 평소 구종 비율과의 차이를 추가한다.
#   merge는 validate="many_to_one"으로 tk_lookup 쪽 키 중복을, 병합 키 dtype을
#   df 쪽에 맞춰 명시적으로 통일해 dtype 불일치로 인한 merge 실패를 방어한다.
#   (v5에서 same_hand를 키에 추가하는 실험도 해봤지만 ablation에서 오히려
#   -1.41 악화돼 기각했다 — 섹션 2.5 참고.)
# global_mean/tk_lookup은 호출부에서 학습 데이터(및 trackman_history)로부터 계산해
# 넘겨주는 상수/정적 테이블이며, 이 함수 안에서 df 자신의 행을 다시 집계하지 않는다.
#
# 이 함수는 notebooks/inference.ipynb, src/script.py에도 문자 그대로(한 글자도
# 다르지 않게) 존재해야 한다 — 학습과 추론이 같은 피처를 만드는지 보장하는 유일한
# 장치이기 때문. 바꾸면 반드시 세 곳 모두 동일하게 반영할 것.

def build_features(df, global_mean, tk_lookup):
    """모델 입력 피처 생성. row_id를 제외한 원본 컬럼 + 파생 피처를 만든다."""
    df = df.copy()

    df["is_two_strike"] = (df["strikes_before"] >= 2).astype(int)
    df["is_three_ball"] = (df["balls_before"] >= 3).astype(int)
    df["is_full_count"] = ((df["balls_before"] >= 3) & (df["strikes_before"] >= 2)).astype(int)
    df["count_diff"] = df["strikes_before"] - df["balls_before"]
    df["count_total"] = df["strikes_before"] + df["balls_before"]

    df["win_exp_diff"] = df["home_win_expectancy"] - df["away_win_expectancy"]
    df["abs_score_diff_pitcher"] = df["score_diff_pitcher_team"].abs()
    df["late_and_close"] = ((df["inning"] >= 8) & (df["abs_score_diff_pitcher"] <= 1)).astype(int)

    df["is_high_leverage"] = (df["li"] >= 1.5).astype(int)
    df["li_count_diff"] = df["li"] * df["count_diff"]
    df["li_late_close"] = df["li"] * df["late_and_close"]

    df["same_hand"] = (df["pitcher_hand"] == df["batter_hand"]).astype(int)

    df["pitcher_cold_start"] = df["asof_pitcher_n"].fillna(0).eq(0).astype(int)
    df["batter_cold_start"] = df["asof_batter_n"].fillna(0).eq(0).astype(int)
    df["pitchmix_cold_start"] = df["asof_pitcher_pitchmix_n"].fillna(0).eq(0).astype(int)

    def shrink(rate_col, n_col, k=30):
        n = df[n_col].fillna(0)
        r = df[rate_col].fillna(global_mean)
        return (n * r + k * global_mean) / (n + k)

    df["pitcher_success_rate_smooth"] = shrink("asof_pitcher_success_rate", "asof_pitcher_n")
    df["batter_success_rate_smooth"] = shrink("asof_batter_success_rate", "asof_batter_n")
    df["matchup_success_diff"] = df["pitcher_success_rate_smooth"] - df["batter_success_rate_smooth"]
    df["matchup_middle_diff"] = (
        df["asof_pitcher_middle_rate"].fillna(global_mean)
        - df["asof_batter_middle_rate"].fillna(global_mean)
    )

    df["pitcher_recent_trend"] = df["asof_pitcher_prev1_game_success_rate"] - df["asof_pitcher_prev5_game_success_rate"]
    df["pitcher_recent_trend3"] = df["asof_pitcher_prev3_game_success_rate"] - df["asof_pitcher_prev5_game_success_rate"]

    fb = df["asof_pitcher_fastball_rate"].fillna(0)
    br = df["asof_pitcher_breaking_rate"].fillna(0)
    os_ = df["asof_pitcher_offspeed_rate"].fillna(0)
    df["pitchmix_max_share"] = np.maximum.reduce([fb, br, os_])

    df["month_sin"] = np.sin(2 * np.pi * df["game_month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["game_month"] / 12)
    df["dow_sin"] = np.sin(2 * np.pi * df["game_dayofweek"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["game_dayofweek"] / 7)

    tk_keys = ["balls_before", "strikes_before", "outs_before"]
    tk_cols = [c for c in tk_lookup.columns if c.startswith("tk_")]
    tk_lookup = tk_lookup[tk_keys + tk_cols].copy()
    for k in tk_keys:
        tk_lookup[k] = tk_lookup[k].astype(df[k].dtype)
    orig_index = df.index
    df = df.merge(
        tk_lookup, on=tk_keys, how="left",
        validate="many_to_one", sort=False,
    )
    df.index = orig_index
    df["tk_fastball_dev"] = fb - df["tk_fastball_rate"]
    df["tk_breaking_dev"] = br - df["tk_breaking_rate"]
    df["tk_offspeed_dev"] = os_ - df["tk_offspeed_rate"]

    return df.drop(columns=[ID_COL], errors="ignore")


# 검증(2024 홀드아웃)에는 2024를 제외하고 만든 TK_LOOKUP_VAL을 쓴다 — 아래 섹션 4에서
# train_fe를 다시 2019~2023(학습)/2024(검증)로 나누므로, 여기서 이미 2024를 안 뺐다고
# 걱정할 필요는 없다: TK_LOOKUP_VAL 자체가 2024 시즌 정보를 전혀 포함하지 않는다.
train_fe = build_features(train, GLOBAL_MEAN_VAL, TK_LOOKUP_VAL)
NEW_COLS = [c for c in train_fe.columns if c not in train.columns]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS] + NEW_COLS
ALL_FEATURES = CAT_COLS + NUM_COLS
for c in CAT_COLS:
    train_fe[c] = train_fe[c].astype("category")

print("파생 피처 수:", len(NEW_COLS), "| 전체 피처 수:", len(ALL_FEATURES))

파생 피처 수: 38 | 전체 피처 수: 85


## 3.5. isotonic 보정용 KFold OOF (v10부터는 스태킹 메타러너 학습에도 재사용)

섹션 6/7의 isotonic 보정기는 "모델이 한 번도 보지 않은 행에 대한 예측"으로 학습해야
보정이 왜곡되지 않는다. issue #4(GPT Pro 4차 리뷰)는 예전 `KFold(shuffle=True)` OOF의
이론적 순환성을 지적했다 — `asof_*` 피처(투수/타자의 시간 누적 성적)가 있는 데이터에서
같은 투수의 이전 투구(A)가 OOF fold에, 이후 투구(B)가 학습 fold에 들어갈 수 있고, B의
`asof_*`에는 A의 실제 결과가 이미 누적돼 있어 A를 예측하는 모델의 학습 데이터 안에
A의 target이 간접적으로 흘러든다는 것이었다(대회 자체의 안티리키지 규칙 위반은 아니다
— 평가 서버 `test.csv`를 들여다보는 게 아니므로). 이를 고치려고 v4~v6은 시즌 단위
expanding-window OOF(`season < Y`로만 학습해 `Y`를 예측)를 썼다.

그런데 이번 v7 조사(로컬 A/B/C 실험, 2024 홀드아웃)에서 expanding-window 방식이
KFold(shuffle) 방식보다 로컬 점수가 뚜렷이 낮고(기준 750.75 대비 KFold 복원이
766.04, +15.29 — v3의 원래 로컬 점수와 정확히 일치), 실제 Dacon 리더보드에서도
KFold 기반이었던 v3(882.11)가 expanding-window 기반인 v4/v5/v6(871~877)보다 계속
높았다는 사실이 확인됐다(project memory
`dacon-leaderboard-regression-investigation` 참고). 즉 expanding-window가 이론상
더 엄밀하지만, fold당 학습 데이터가 줄어들고(특히 초기 시즌 fold) 최신성이 떨어지는
비용이 이 경미한 순환성을 고치는 이득보다 커 보인다는 것이 로컬·실측 양쪽에서
일관되게 나타났다.

그래서 v7은 **보정용 OOF 생성 메커니즘만** 원래의 KFold(shuffle=True, n_splits=5,
random_state=42)로 되돌린다 — `TK_LOOKUP_VAL`/`TK_LOOKUP_FULL`의 시즌 분리(섹션 2.5,
진짜 리크 방지 로직)나 `build_features()`, 블렌드 가중치/순서 탐색 인프라는 전혀
건드리지 않는다. 이미 만들어진 `X_train`/`X_train_cb`/`y_train`(섹션 4, 검증용)과
`X_full`/`X_full_cb`/`y_full`(섹션 7, 전체 데이터용)을 그대로 KFold로 나눠 학습/예측한다
— fold마다 `global_mean`/`tk_lookup`을 다시 계산하지 않는다(이미 섹션 4/7에서 고정된
`TK_LOOKUP_VAL`/`TK_LOOKUP_FULL` 기준으로 피처가 만들어져 있기 때문). v5부터 이어온
대로 LightGBM과 CatBoost의 OOF 예측은 블렌딩하지 않고 **따로따로** 반환한다 — 섹션
6에서 블렌드 가중치와 보정 순서를 격자 탐색할 때 모델을 다시 학습하지 않고 이미 계산된
`lgb_oof`/`cb_oof` 값만 재조합하면 되도록 하기 위함이다.

**v10에서는 이 동일한 KFold OOF를 블렌드 가중치 격자탐색이 아니라 로지스틱 회귀
메타러너(스태킹)의 학습 입력으로 쓴다** — 섹션 6 참고. 이 재사용의 위험성(메타러너
학습에 calibration과 동일한 OOF를 그대로 쓰는 것)은 섹션 6에서 별도로 논의한다.

**이 PR의 진짜 통과 기준은 로컬 홀드아웃 점수가 아니라 Dacon 재제출 실측 점수다** — 지난
v4/v5/v6 모두 로컬은 개선됐지만 실제 리더보드는 악화된 전례가 있으므로, 이번 로컬
재현(766 근방)은 "가설이 강하게 뒷받침됨" 정도로만 받아들이고 최종 판단은 재제출
후로 미룬다.


In [5]:
def kfold_oof(X, X_cb, y, lgb_n_estimators, cb_iterations, cat_idx,
              n_splits=5, random_state=42):
    """KFold(shuffle=True) 기반 isotonic 보정용 OOF 예측 생성 (LightGBM/CatBoost 따로 반환).

    X/y 전체를 n_splits개 fold로 무작위 분할해, 각 fold는 나머지 fold로 학습한 모델의
    예측값으로 채운다. X/X_cb는 이미 build_features()로 피처가 만들어지고
    (섹션 4/7에서 각각 TK_LOOKUP_VAL/TK_LOOKUP_FULL 기준으로 생성됨) ALL_FEATURES로
    정렬된 상태여야 한다 — 이 함수는 global_mean/tk_lookup을 다시 계산하지 않는다.
    반환값은 (lgb_oof, cb_oof, y_oof) — 블렌딩은 호출부에서 한다.
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    X_r = X.reset_index(drop=True)
    X_cb_r = X_cb.reset_index(drop=True)
    y_r = y.reset_index(drop=True)

    lgb_oof = np.zeros(len(X_r))
    cb_oof = np.zeros(len(X_r))
    for fold_i, (tr_idx, oof_idx) in enumerate(kf.split(X_r)):
        ft = time.time()
        m_lgb = lgb.LGBMClassifier(n_estimators=lgb_n_estimators, **LGB_PARAMS)
        m_lgb.fit(X_r.iloc[tr_idx], y_r.iloc[tr_idx])
        lgb_oof[oof_idx] = m_lgb.predict_proba(X_r.iloc[oof_idx])[:, 1]

        tp = Pool(X_cb_r.iloc[tr_idx], y_r.iloc[tr_idx], cat_features=cat_idx)
        m_cb = CatBoostClassifier(iterations=cb_iterations, **CB_PARAMS)
        m_cb.fit(tp)
        cb_oof[oof_idx] = m_cb.predict_proba(X_cb_r.iloc[oof_idx])[:, 1]
        print(f"  KFold {fold_i + 1}/{n_splits} 완료 :: {time.time() - ft:.1f}s")

    return lgb_oof, cb_oof, y_r.to_numpy()


## 4. 모델 학습 및 검증 (LightGBM + CatBoost)

LightGBM과 CatBoost 둘 다 범주형 3개(`top_bottom`, `game_type`, `base_state`)를
네이티브로 처리하고 결측값도 자동으로 다루므로 별도의 `OrdinalEncoder`/`SimpleImputer`
파이프라인이 필요 없습니다. (CatBoost는 범주형 컬럼을 문자열로 넘겨야 해서
`X_*_cb` 사본을 따로 만듭니다.)

2024 시즌을 검증용으로 떼어 두고 2019~2023으로 학습하며, 두 모델 모두
`early_stopping`으로 트리 개수(`best_iteration`)를 찾습니다. 이 값들은 "전체 데이터
재학습" 단계에서 고정 라운드 수로 재사용합니다.

In [6]:
LGB_PARAMS = dict(
    learning_rate=0.03, num_leaves=63, min_child_samples=200,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    objective="binary", random_state=42, n_jobs=-1, verbosity=-1,
)
CB_PARAMS = dict(
    learning_rate=0.03, depth=8, l2_leaf_reg=3.0,
    loss_function="Logloss", random_seed=42, verbose=False, thread_count=-1,
)

is_val = train_fe["season"] == 2024
X_train, y_train = train_fe.loc[~is_val, ALL_FEATURES], train_fe.loc[~is_val, TARGET]
X_val, y_val = train_fe.loc[is_val, ALL_FEATURES], train_fe.loc[is_val, TARGET]
print("train:", len(X_train), "| val:", len(X_val))

cat_idx = [ALL_FEATURES.index(c) for c in CAT_COLS]
X_train_cb = X_train.copy(); X_val_cb = X_val.copy()
for c in CAT_COLS:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_val_cb[c] = X_val_cb[c].astype(str)

t = time.time()
val_model = lgb.LGBMClassifier(n_estimators=2000, **LGB_PARAMS)
val_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
BEST_ITERATION_LGB = val_model.best_iteration_
print(f"LightGBM 학습 완료 :: {time.time() - t:.1f}s | best_iteration={BEST_ITERATION_LGB}")

t = time.time()
train_pool = Pool(X_train_cb, y_train, cat_features=cat_idx)
val_pool = Pool(X_val_cb, y_val, cat_features=cat_idx)
cb_val_model = CatBoostClassifier(iterations=3000, early_stopping_rounds=50, **CB_PARAMS)
cb_val_model.fit(train_pool, eval_set=val_pool, use_best_model=True)
# CatBoost의 get_best_iteration()은 0-based 인덱스라, use_best_model=True로 실제 선택된
# 모델의 트리 개수(tree_count_)는 이 값 + 1이다. 이후 CatBoostClassifier(iterations=...)로
# 같은 트리 개수를 재현하려면 +1을 반드시 더해야 한다 (직접 검증: get_best_iteration()=145일
# 때 tree_count_=146). LightGBM의 best_iteration_은 이미 카운트라 이 보정이 필요 없다.
BEST_ITERATION_CB = cb_val_model.get_best_iteration() + 1
print(f"CatBoost 학습 완료 :: {time.time() - t:.1f}s | best_iteration={BEST_ITERATION_CB}")

train: 1221585 | val: 253507


LightGBM 학습 완료 :: 7.4s | best_iteration=164


CatBoost 학습 완료 :: 82.1s | best_iteration=436


## 5. 검증 — LightGBM/CatBoost 단독 Brier Skill Score

학습 데이터에서 떼어 둔 2024 시즌으로 LightGBM 단독, CatBoost 단독 예측의 검증
점수를 각각 계산한다 (참고용 — 실제 블렌드 가중치와 보정 순서는 섹션 6에서
데이터 기반으로 고른다).

Brier 는 예측 확률과 실제값(0/1) 차이의 제곱 평균이고, 이를 상수 예측의 Brier 인
`r(1-r)` 로 나누어 Brier Skill Score 를 구한다.

In [7]:
def brier_score(y_true, p):
    r = y_true.mean()
    brier = ((p - y_true) ** 2).mean()
    baseline_brier = r * (1 - r)
    return max(0, 100000 * (1 - brier / baseline_brier)), brier


val_pred_lgb = val_model.predict_proba(X_val)[:, 1]
val_pred_cb = cb_val_model.predict_proba(X_val_cb)[:, 1]

s_lgb, b_lgb = brier_score(y_val, val_pred_lgb)
s_cb, b_cb = brier_score(y_val, val_pred_cb)
print(f"LightGBM 단독:  score={s_lgb:.2f} (brier={b_lgb:.6f})")
print(f"CatBoost 단독:  score={s_cb:.2f} (brier={b_cb:.6f})")

LightGBM 단독:  score=720.59 (brier=0.248007)
CatBoost 단독:  score=740.89 (brier=0.247956)


In [8]:
# =======================
# 6. 로지스틱 회귀 메타러너 스태킹 (KFold OOF)
# =======================
# v10부터는 "블렌드 가중치 격자탐색(W_GRID) + 보정 순서 격자탐색"(v5~v9)을 LightGBM/
# CatBoost의 KFold OOF 예측 2개를 피처로 하는 로지스틱 회귀 메타러너로 교체한다(stacking).
#
# OOF 재사용에 대한 판단: 메타러너 학습에 쓰는 [lgb_oof, cb_oof] -> y_oof는 이전까지
# isotonic 보정에 쓰던 것과 동일한 KFold(shuffle=True, 5-fold) OOF다. scikit-learn
# StackingClassifier 문서의 권고대로 메타러너는 OOF(in-sample이 아닌) 예측으로
# 학습해야 하며, 피처가 2개뿐인 선형 로지스틱 회귀는 표현력이 낮아(사실상 비음수/
# 합=1 제약이 없는 가중 블렌드의 일반화판) 과적합 위험이 낮다고 판단해 별도의 중첩
# KFold를 두지 않았다 — GBM 등 고용량 메타러너였다면 이 판단은 달랐을 것이다
# (docs/logistic_stack_diagnostic.py 참고).
#
# raw 출력과 isotonic 보정 후 출력을 모두 실험한 결과(2024 홀드아웃):
#   메타러너 raw(보정 없음)         782.47
#   메타러너 + isotonic 보정 1개 더  770.21  (-12.26, 기존 W_GRID+isotonic 기준 774.07보다도 낮음)
# "선형 스태킹은 베이스보다 더 날카로운 출력을 내서 오히려 보정을 해칠 수 있다"는
# 가설이 실측됐다 — 그래서 v10은 메타러너의 raw predict_proba를 그대로 쓰고 추가
# 보정기를 씌우지 않는다.

t = time.time()
lgb_oof, cb_oof, y_oof = kfold_oof(
    X_train, X_train_cb, y_train, BEST_ITERATION_LGB, BEST_ITERATION_CB, cat_idx,
)
print(f"OOF 생성 완료 :: {time.time() - t:.1f}s")

X_meta_oof = np.column_stack([lgb_oof, cb_oof])
stack_model = LogisticRegression()
stack_model.fit(X_meta_oof, y_oof)
print(f"메타러너 계수: coef={stack_model.coef_}, intercept={stack_model.intercept_}")

X_meta_val = np.column_stack([val_pred_lgb, val_pred_cb])
val_pred_final = stack_model.predict_proba(X_meta_val)[:, 1]

s_cal, b_cal = brier_score(y_val, val_pred_final)
print(f"Validation Score (로지스틱 스태킹, raw): {s_cal:.2f} (brier={b_cal:.6f})")

# 참고용 — isotonic 보정을 얹었을 때(기각된 경로)와 비교. 최종 아티팩트/제출에는 쓰지 않는다.
from sklearn.isotonic import IsotonicRegression as _IsotonicRegression_diag
_stack_pred_oof_diag = stack_model.predict_proba(X_meta_oof)[:, 1]
_stack_cal_diag = _IsotonicRegression_diag(out_of_bounds="clip").fit(_stack_pred_oof_diag, y_oof)
_s_stack_cal_diag, _ = brier_score(y_val, _stack_cal_diag.predict(val_pred_final))
print(f"(참고, 미채택) 로지스틱 스태킹 + isotonic 보정: {_s_stack_cal_diag:.2f}")

  KFold 1/5 완료 :: 73.4s


  KFold 2/5 완료 :: 73.7s


  KFold 3/5 완료 :: 73.0s


  KFold 4/5 완료 :: 74.1s


  KFold 5/5 완료 :: 73.4s
OOF 생성 완료 :: 367.8s
메타러너 계수: coef=[[2.32650526 2.2717728 ]], intercept=[-2.31331817]
Validation Score (로지스틱 스태킹, raw): 782.47 (brier=0.247852)


/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/admin/Project/LG-A

(참고, 미채택) 로지스틱 스태킹 + isotonic 보정: 770.21


## 7. 전체 데이터로 재학습, 메타러너 재생성 & 모델 저장

검증으로 성능을 확인했으니 이제 전체 학습 데이터(2019~2024)로 LightGBM과 CatBoost를
각각 다시 학습합니다. 라운드 수는 검증 단계에서 찾은 `BEST_ITERATION_LGB`/
`BEST_ITERATION_CB`로 고정합니다(전체 데이터에는 2025시즌 같은 미래 검증 세트가 없어
early stopping을 다시 돌릴 수 없기 때문).

로지스틱 회귀 메타러너도 전체 데이터 기준 KFold(shuffle=True, 5-fold) OOF 예측(섹션
3.5/6과 동일한 방식)으로 다시 학습합니다 — 검증 단계의 메타러너(2019~2023만 사용)보다
2024를 포함한 전체 데이터 분포를 반영하므로 2025 평가 데이터에 더 가깝습니다. 섹션
6에서 확인한 대로 추가 isotonic 보정은 씌우지 않습니다.

학습한 두 모델·로지스틱 회귀 메타러너(`stack_model`)·피처 목록·global_mean을 하나의
딕셔너리로 묶어 `./model/ensemble.pkl` 로 저장합니다. 이 파일을 추론용 `script.py`,
`requirements.txt` 와 함께 제출용 zip으로 묶으면 제출 준비가 끝납니다.

In [9]:
GLOBAL_MEAN_FULL = float(train[TARGET].mean())
train_fe_full = build_features(train, GLOBAL_MEAN_FULL, TK_LOOKUP_FULL)
for c in CAT_COLS:
    train_fe_full[c] = train_fe_full[c].astype("category")
X_full = train_fe_full[ALL_FEATURES]
y_full = train_fe_full[TARGET]
X_full_cb = X_full.copy()
for c in CAT_COLS:
    X_full_cb[c] = X_full_cb[c].astype(str)

t = time.time()
final_lgb = lgb.LGBMClassifier(n_estimators=BEST_ITERATION_LGB, **LGB_PARAMS)
final_lgb.fit(X_full, y_full)
print(f"LightGBM 전체 재학습 완료 :: {time.time() - t:.1f}s")

t = time.time()
full_pool = Pool(X_full_cb, y_full, cat_features=cat_idx)
final_cb = CatBoostClassifier(iterations=BEST_ITERATION_CB, **CB_PARAMS)
final_cb.fit(full_pool)
print(f"CatBoost 전체 재학습 완료 :: {time.time() - t:.1f}s")

t = time.time()
lgb_full_oof, cb_full_oof, y_full_oof = kfold_oof(
    X_full, X_full_cb, y_full, BEST_ITERATION_LGB, BEST_ITERATION_CB, cat_idx,
)

# 섹션 6에서 확인한 대로 로지스틱 회귀 메타러너를 전체 데이터 OOF로 다시 학습한다.
# 추가 isotonic 보정은 씌우지 않는다(섹션 6 ablation: raw 782.47 vs 보정후 770.21).
X_meta_full_oof = np.column_stack([lgb_full_oof, cb_full_oof])
final_stack_model = LogisticRegression()
final_stack_model.fit(X_meta_full_oof, y_full_oof)
oof_pred_for_report = final_stack_model.predict_proba(X_meta_full_oof)[:, 1]
print(f"전체 데이터 OOF 메타러너 학습 완료 :: {time.time() - t:.1f}s")
print(f"메타러너 계수(전체데이터): coef={final_stack_model.coef_}, intercept={final_stack_model.intercept_}")

s_full, b_full = brier_score(y_full_oof, oof_pred_for_report)
print(f"전체 데이터 OOF 스태킹 점수(참고용, in-sample): {s_full:.2f} (brier={b_full:.6f})")

os.makedirs("../model", exist_ok=True)
artifact = {
    "lgbm_model": final_lgb,
    "catboost_model": final_cb,
    "stack_model": final_stack_model,
    "cat_cols": CAT_COLS,
    "all_features": ALL_FEATURES,
    "global_mean": GLOBAL_MEAN_FULL,
    "tk_lookup": TK_LOOKUP_FULL,
    "lgb_params": {**LGB_PARAMS, "n_estimators": BEST_ITERATION_LGB},
    "cb_params": {**CB_PARAMS, "iterations": BEST_ITERATION_CB},
}
joblib.dump(artifact, "../model/ensemble.pkl", compress=3)
print("저장 완료: ../model/ensemble.pkl")

LightGBM 전체 재학습 완료 :: 9.9s


CatBoost 전체 재학습 완료 :: 97.5s


  KFold 1/5 완료 :: 83.1s


  KFold 2/5 완료 :: 78.3s


  KFold 3/5 완료 :: 74.8s


  KFold 4/5 완료 :: 74.9s


  KFold 5/5 완료 :: 75.1s
전체 데이터 OOF 메타러너 학습 완료 :: 386.6s
메타러너 계수(전체데이터): coef=[[2.29974248 2.25457221]], intercept=[-2.28689433]
전체 데이터 OOF 스태킹 점수(참고용, in-sample): 1891.57 (brier=0.244717)
저장 완료: ../model/ensemble.pkl


/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/admin/Project/LG-Aimers-Claude/.venv311/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/admin/Project/LG-A